In [1]:
!pip install imbalance-metrics smogn

In [2]:
!pip install ImbalancedLearningRegression


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.2/77.2 kB 1.7 MB/s eta 0:00:00


In [3]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import PchipInterpolator
from smogn import phi, phi_ctrl_pts
from sklearn.model_selection import KFold
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, BaggingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor
import seaborn as sns
from sklearn.neighbors import KNeighborsRegressor
from scipy.spatial import distance
from scipy.spatial import distance_matrix
from scipy.sparse.csgraph import minimum_spanning_tree
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    AdaBoostRegressor,
    BaggingRegressor
)
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.linear_model import (
    ARDRegression,
    SGDRegressor,
    PassiveAggressiveRegressor
)
from sklearn.kernel_ridge import KernelRidge
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from smogn import smoter
import ImbalancedLearningRegression as iblr

In [4]:
models_pool = [
    SVR(),
    RandomForestRegressor(n_estimators=100, random_state=42),
    DecisionTreeRegressor(random_state=42),
    MLPRegressor(hidden_layer_sizes=(50,50), max_iter=1000, random_state=42),
    BaggingRegressor(random_state=42),
    XGBRegressor(n_estimators=100, random_state=42)
]

# models_pool = [
#     RandomForestRegressor(n_estimators=100, random_state=42),
#     ExtraTreesRegressor(n_estimators=100, random_state=42),
#     XGBRegressor(n_estimators=100, random_state=42)
# ]

In [5]:
def instance_hardness_regression(X, y, models=models_pool, distance='l1', n_splits=10, logs=False, gamma_fixo = False, fator_divisao_gamma = 1):
    n = len(y)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    if gamma_fixo:
      gamma = 1
    else:
      ### Justamente o que diz no artigo, que é a media do target elevado a 2
      gamma = np.mean(y**2) / fator_divisao_gamma #### Gamma menor -> Maior sensibilidade a erros, deixando o valor final de IG mais proximo de 1 ( Por padrão é np.mean(y**2))

    preds_pool = np.zeros((n, len(models)))

    ### Cross-validation ###
    for fold_idx, (train_idx, test_idx) in enumerate(kf.split(X), start=1):
        if logs:
          print(f"Iniciando fold {fold_idx}/{n_splits}...")
        X_train, X_test = X[train_idx], X[test_idx]
        y_train = y[train_idx]

        ### ADICIONAR OS TIPOS DE BALANCEAMENTO NO X_TRAIN AQUI
        for j, model in enumerate(models):
            if logs:
              print(f"Treinando modelo {j+1}/{len(models)}: {type(model).__name__} ...")
            clone_model = type(model)(**model.get_params())
            clone_model.fit(X_train, y_train)
            preds_pool[test_idx, j] = clone_model.predict(X_test)

    if distance == 'l1':
        dists = np.abs(y.reshape(-1, 1) - preds_pool)
    elif distance == 'l2':
        dists = (y.reshape(-1, 1) - preds_pool)**2
    else:
        raise ValueError("Use 'l1' ou 'l2'.")

    exp_term = np.exp(-dists / gamma)
    ih_values = 1 - np.mean(exp_term, axis=1) ### Isso equivale a divisão por |L|

    return ih_values

In [14]:
import numpy as np

def distance(y, D=1.0, delta=1.0):
    """
    Distance-based relevance function φ_D
    y: array-like (assumido padronizado)
    """
    y = np.asarray(y)
    n = len(y)
    raw_scores = np.zeros(n)

    for j in range(n):
        s = 0.0
        for i in range(n):
            d = abs(y[j] - y[i])
            if 0 < d < D:
                s += 1.0 / (d + delta)
        raw_scores[j] = s

    s_min = raw_scores.min()
    s_max = raw_scores.max()

    if s_max == s_min:
        raise ValueError("redefine relevance function: all points have same density")

    dist = (s_max - raw_scores) / (s_max - s_min)
    return dist


In [15]:
## load dependencies - third party
import numpy as np
import pandas as pd
import random as rd

## generate synthetic observations
def over_sampling_gn(

    ## arguments / inputs
    data,       ## training set
    index,      ## index of input data
    perc,       ## over / under sampling
    pert,       ## perturbation / noise percentage
    replace,    ## sampling replacement (bool)

    ):

    """
    generates synthetic observations and is the primary function underlying the
    over-sampling technique utilized in the higher main function 'gn()', the
    4 step procedure for generating synthetic observations is:

    1) pre-processing: temporarily removes features without variation, label
    encodes nominal / categorical features, and subsets the training set into
    two data sets by data type: numeric / continuous, and nominal / categorical

    2) over-sampling: GN, which perturb the interpolated values with gaussian noise

    3) post processing: restores original values for label encoded features,
    reintroduces constant features previously removed, converts any interpolated
    negative values to zero in the case of non-negative features

    returns a pandas dataframe containing synthetic observations of the training
    set which are then returned to the higher main function 'gn()'

    ref:

    Branco, P., Torgo, L., Ribeiro, R. (2017).
    SMOGN: A Pre-Processing Approach for Imbalanced Regression.
    Proceedings of Machine Learning Research, 74:36-50.
    http://proceedings.mlr.press/v74/branco17a/branco17a.pdf.

    Branco, P., Ribeiro, R., Torgo, L. (2017).
    Package 'UBL'. The Comprehensive R Archive Network (CRAN).
    https://cran.r-project.org/web/packages/UBL/UBL.pdf.

    Branco, P., Torgo, L., & Ribeiro, R. P. (2019).
    Pre-processing approaches for imbalanced distributions in regression.
    Neurocomputing, 343, 76-99.
    https://www.sciencedirect.com/science/article/abs/pii/S0925231219301638

    Kunz, N., (2019). SMOGN.
    https://github.com/nickkunz/smogn
    """

    ## subset original dataframe by bump classification index
    data = data.iloc[index]

    ## store dimensions of data subset
    n = len(data)
    d = len(data.columns)

    ## store original data types
    feat_dtypes_orig = [None] * d

    for j in range(d):
        feat_dtypes_orig[j] = data.iloc[:, j].dtype

    ## find non-negative numeric features
    feat_non_neg = []
    num_dtypes = ["int64", "float64"]

    for j in range(d):
        if data.iloc[:, j].dtype in num_dtypes and any(data.iloc[:, j] > 0):
            feat_non_neg.append(j)

    ## find features without variation (constant features)
    feat_const = data.columns[data.nunique() == 1]

    ## temporarily remove constant features
    if len(feat_const) > 0:

        ## create copy of orignal data and omit constant features
        data_orig = data.copy()
        data = data.drop(data.columns[feat_const], axis = 1)

        ## store list of features with variation
        feat_var = list(data.columns.values)

        ## reindex features with variation
        for i in range(d - len(feat_const)):
            data.rename(columns = {
                data.columns[i]: i
                }, inplace = True)

        ## store new dimension of feature space
        d = len(data.columns)

    ## create copy of data containing variation
    data_var = data.copy()

    ## create global feature list by column index
    feat_list = list(data.columns.values)

    ## create nominal feature list and
    ## label encode nominal / categorical features
    ## (strictly label encode, not one hot encode)
    feat_list_nom = []
    nom_dtypes = ["object", "bool", "datetime64"]

    # Unknown warning, may be handled later
    pd.options.mode.chained_assignment = None

    for j in range(d):
        if data.dtypes[j] in nom_dtypes:
            feat_list_nom.append(j)
            data.iloc[:, j] = pd.Categorical(pd.factorize(
                data.iloc[:, j])[0])

    data = data.apply(pd.to_numeric)

    ## create numeric feature list
    feat_list_num = list(set(feat_list) - set(feat_list_nom))

    ## calculate ranges for numeric / continuous features
    ## (includes label encoded features)
    feat_ranges = list(np.repeat(1, d))

    if len(feat_list_nom) > 0:
        for j in feat_list_num:
            feat_ranges[j] = max(data.iloc[:, j]) - min(data.iloc[:, j])
    else:
        for j in range(d):
            feat_ranges[j] = max(data.iloc[:, j]) - min(data.iloc[:, j])

    ## subset feature ranges to include only numeric features
    ## (excludes label encoded features)
    feat_ranges_num = [feat_ranges[i] for i in feat_list_num]

    ## subset data by either numeric / continuous or nominal / categorical
    data_num = data.iloc[:, feat_list_num]
    data_nom = data.iloc[:, feat_list_nom]

    ## get number of features for each data type
    feat_count_num = len(feat_list_num)
    feat_count_nom = len(feat_list_nom)


    ## number of new synthetic observations for each rare observation
    x_synth = int(perc - 1)

    ## total number of new synthetic observations to generate
    n_synth = int(n * (perc - 1 - x_synth))

    ## randomly index data by the number of new synthetic observations
    r_index = np.random.choice(
        a = tuple(range(0, n)),
        size = n_synth,
        replace = replace,
        p = None
    )

    ## create null matrix to store new synthetic observations
    synth_matrix = np.ndarray(shape = ((x_synth * n + n_synth), d))

    # modified
    if x_synth > 0:
        for i in range(n):
            for j in range(x_synth):
                for attr in range(d):
                    if pd.isna(data.iloc[i, attr]):
                            synth_matrix[i * x_synth + j, attr] = None
                    if attr in feat_list_nom:
                        synth_matrix[i * x_synth + j, attr] = np.random.choice(
                            a=list(data.iloc[:, attr]))
                    else:
                        synth_matrix[i * x_synth + j, attr] = data.iloc[
                            i, attr] + np.random.normal(0,
                            np.sqrt(pert * np.std(list(data.iloc[:, attr]))))

    # modified
    if n_synth > 0:
        count = 0
        for i in r_index:
            for attr in range(d):
                    if pd.isna(data.iloc[i, attr]):
                            synth_matrix[x_synth * n + count, attr] = None
                    if attr in feat_list_nom:
                        synth_matrix[x_synth * n + count, attr] = np.random.choice(
                            a=list(data.iloc[:, attr]))
                    else:
                        synth_matrix[x_synth * n + count, attr] = data.iloc[
                            i, attr] + np.random.normal(0,
                            np.sqrt(pert * np.std(list(data.iloc[:, attr]))))
            ## close loop counter
            count = count + 1

    ## convert synthetic matrix to dataframe
    data_new = pd.DataFrame(synth_matrix)

    ## synthetic data quality check
    if sum(data_new.isnull().sum()) > 0:
        raise ValueError("oops! synthetic data contains missing values")

    ## replace label encoded values with original values
    for j in feat_list_nom:
        code_list = data.iloc[:, j].unique()
        cat_list = data_var.iloc[:, j].unique()

        for x in code_list:
            data_new.iloc[:, j] = data_new.iloc[:, j].replace(x, cat_list[x])

    ## reintroduce constant features previously removed
    if len(feat_const) > 0:
        data_new.columns = feat_var

        for j in range(len(feat_const)):
            data_new.insert(
                loc = int(feat_const[j]),
                column = feat_const[j],
                value = np.repeat(
                    data_orig.iloc[0, feat_const[j]],
                    len(synth_matrix))
            )

    ## convert negative values to zero in non-negative features
    for j in feat_non_neg:
        # data_new.iloc[:, j][data_new.iloc[:, j] < 0] = 0
        data_new.iloc[:, j] = data_new.iloc[:, j].clip(lower = 0)

    ## return over-sampling results dataframe
    return data_new

In [24]:
def apply_gn_dist(

    ## main arguments / inputs
    data,                     ## training set (pandas dataframe)
    y,                        ## response variable y by name (string)
    pert = 0.02,              ## perturbation / noise percentage (pos real)
    samp_method = "balance",  ## over / under sampling ("balance" or extreme")
    under_samp = True,        ## under sampling (bool)
    drop_na_col = True,       ## auto drop columns with nan's (bool)
    drop_na_row = True,       ## auto drop rows with nan's (bool)
    replace = False,          ## sampling replacement (bool)
    manual_perc = False,      ## user defines percentage of under-sampling and over-sampling  # added
    perc_u = -1,              ## percentage of under-sampling  # added
    perc_o = -1,              ## percentage of over-sampling  # added

    ## phi relevance function arguments / inputs
    rel_thres = 0.5,          ## relevance threshold considered rare (pos real)
    rel_method = "auto",      ## relevance method ("auto" or "manual")
    rel_xtrm_type = "both",   ## distribution focus ("high", "low", "both")
    rel_coef = 1.5,           ## coefficient for box plot (pos real)
    rel_ctrl_pts_rg = None    ## input for "manual" rel method  (2d array)

    ):

    """
    the main function, designed to help solve the problem of imbalanced data
    for regression; GN applies the combintation of under-sampling the majority
    class (in the case of regression, values commonly found near the mean of
    a normal distribution in the response variable y) and over-sampling the
    minority class (rare values in a normal distribution of y, typically found
    at the tails)

    procedure begins with a series of pre-processing steps, and to ensure no
    missing values (nan's), sorts the values in the response variable y by
    ascending order, and fits a function 'phi' to y, corresponding phi values
    (between 0 and 1) are generated for each value in y, the phi values are
    then used to determine if an observation is either normal or rare by the
    threshold specified in the argument 'rel_thres'

    normal observations are placed into a majority class subset (normal bin)
    and are under-sampled, while rare observations are placed in a seperate
    minority class subset (rare bin) where they're over-sampled

    under-sampling is applied by a random sampling from the normal bin based
    on a calculated percentage control by the argument 'samp_method', if the
    specified input of 'samp_method' is "balance", less under-sampling (and
    over-sampling) is conducted, and if "extreme" is specified more under-
    sampling (and over-sampling is conducted)

    over-sampling is applied by GN, which perturb the interpolated values
    with gaussian noise

    GN is only applied to numeric / continuous features, synthetic values
    found in nominal / categorical features, are generated by randomly
    selecting observed values found within their respective feature

    procedure concludes by post-processing and returns a modified pandas data
    frame containing under-sampled and over-sampled (synthetic) observations,
    the distribution of the response variable y should more appropriately
    reflect the minority class areas of interest in y that are under-
    represented in the original training set

    note that users can also decide the percentage of under-sampling and
    over-sampling on each bin by setting manual_perc to True and assigning
    values to perc_u and perc_o, these values will directly replace the
    calculated percentage of each bin

    ref:

    Branco, P., Torgo, L., Ribeiro, R. (2017).
    SMOGN: A Pre-Processing Approach for Imbalanced Regression.
    Proceedings of Machine Learning Research, 74:36-50.
    http://proceedings.mlr.press/v74/branco17a/branco17a.pdf.

    Branco, P., Torgo, L., & Ribeiro, R. P. (2019).
    Pre-processing approaches for imbalanced distributions in regression.
    Neurocomputing, 343, 76-99.
    https://www.sciencedirect.com/science/article/abs/pii/S0925231219301638

    Kunz, N., (2019). SMOGN.
    https://github.com/nickkunz/smogn
    """

    ## pre-process missing values
    if bool(drop_na_col) == True:
        data = data.dropna(axis = 1)  ## drop columns with nan's

    if bool(drop_na_row) == True:
        data = data.dropna(axis = 0)  ## drop rows with nan's

    ## quality check for missing values in dataframe
    if data.isnull().values.any():
        raise ValueError("cannot proceed: data cannot contain NaN values")

    ## quality check for y
    if isinstance(y, str) is False:
        raise ValueError("cannot proceed: y must be a string")

    if y in data.columns.values is False:
        raise ValueError("cannot proceed: y must be an header name (string) \
               found in the dataframe")

    ## quality check for perturbation
    if pert > 1 or pert <= 0:
        raise ValueError("pert must be a real number number: 0 < R < 1")

    ## quality check for sampling method
    if samp_method in ["balance", "extreme"] is False:
        raise ValueError("samp_method must be either: 'balance' or 'extreme'")

    # added
    ## quality check for sampling percentage
    if manual_perc:
        if perc_u == -1 or perc_o == -1:
            raise ValueError("cannot proceed: require percentage of under-sampling and over-sampling if manual_perc == True")
        if perc_u >= 1 or perc_u <= 0:
            raise ValueError("percentage of under-sampling must be a real number between 0 and 1")
        if perc_o <= 0:
            raise ValueError("percentage of over-sampling must be a positve real number")

    ## quality check for relevance threshold parameter
    if rel_thres == None:
        raise ValueError("cannot proceed: relevance threshold required")

    if rel_thres > 1 or rel_thres <= 0:
        raise ValueError("rel_thres must be a real number number: 0 < R < 1")

    ## store data dimensions
    n = len(data)
    d = len(data.columns)

    ## store original data types
    feat_dtypes_orig = [data.iloc[:, j].dtype for j in range(d)]

    original_index = data.index.copy()

    # -------------------------
    # mover y para última coluna
    # -------------------------
    y_col = data.columns.get_loc(y)
    if y_col < d - 1:
        cols = list(range(d))
        cols[y_col], cols[d - 1] = cols[d - 1], cols[y_col]
        data = data[data.columns[cols]]

    feat_names = list(data.columns)
    data.columns = range(d)

    # -------------------------
    # ordenar Y
    # -------------------------
    y_df = pd.DataFrame(data[d - 1])
    y_sort = y_df.sort_values(by=d - 1)
    y_sort_series = y_sort[d - 1]

    # =========================================================
    # 🔁 SUBSTITUIÇÃO DO IH POR φᴰ (APENAS Y)
    # =========================================================

    y_sorted_values = y_sort_series.values

    # padronização
    y_std = (y_sorted_values - np.mean(y_sorted_values)) / np.std(y_sorted_values)

    # cálculo da relevância φᴰ
    dist_y = distance(
        y=y_std,
        D=1.0,
        delta=1.0
    )

    if np.all(dist_y == 0):
        raise ValueError("redefine relevance function: all points are 0")

    if np.all(dist_y == 1):
        raise ValueError("redefine relevance function: all points are 1")

    # -------------------------
    # determinar bumps (φᴰ > 0.5)
    # -------------------------
    bumps = [0]
    rel_thres = 0.5

    for i in range(len(dist_y) - 1):
        if ((dist_y[i] >= rel_thres and dist_y[i + 1] < rel_thres) or
            (dist_y[i] < rel_thres and dist_y[i + 1] >= rel_thres)):
            bumps.append(i + 1)

    bumps.append(n)
    n_bumps = len(bumps) - 1

    ## determine indicies for each bump classification
    b_index = {}

    for i in range(n_bumps):
        b_index.update({i: y_sort[bumps[i]:bumps[i + 1]]})

    ## calculate over / under sampling percentage according to
    ## bump class and user specified method ("balance" or "extreme")
    b = round(n / n_bumps)
    s_perc = []
    scale = []
    obj = []

    if samp_method == "balance":
        for i in b_index:
            s_perc.append(b / len(b_index[i]))

    if samp_method == "extreme":
        for i in b_index:
            scale.append(b ** 2 / len(b_index[i]))
        scale = n_bumps * b / sum(scale)

        for i in b_index:
            obj.append(round(b ** 2 / len(b_index[i]) * scale, 2))
            s_perc.append(round(obj[i] / len(b_index[i]), 1))

    ## conduct over / under sampling and store modified training set
    data_new = pd.DataFrame()

    for i in range(n_bumps):

        ## no sampling
        if s_perc[i] == 1:

            ## simply return no sampling
            ## results to modified training set
            data_new = pd.concat([data.iloc[b_index[i].index], data_new], ignore_index = True)

        ## over-sampling
        if s_perc[i] > 1:

            ## generate synthetic observations in training set
            ## considered 'minority'
            ## (see 'over_sampling_gn()' function for details)
            synth_obs = over_sampling_gn(
                data = data,
                index = list(b_index[i].index),
                perc = s_perc[i] if not manual_perc else perc_o + 1,  # modified
                pert = pert,
                replace = replace  # added
            )

            ## concatenate over-sampling
            ## results to modified training set
            data_new = pd.concat([synth_obs, data_new], ignore_index = True)

            # added
            ## concatenate original data
            ## to modified training set
            original_obs = data.iloc[list(b_index[i].index)]
            data_new = pd.concat([original_obs, data_new], ignore_index = True)

        ## under-sampling
        if s_perc[i] < 1:  # exchanged

            if under_samp is True:  # exchanged

                ## drop observations in training set
                ## considered 'normal' (not 'rare')
                chosen_index = np.random.choice(  # modified
                    a = list(b_index[i].index),
                    size = int((s_perc[i] if not manual_perc else perc_u) * len(b_index[i])),
                    replace = replace
                )

                chosen_obs = data.iloc[chosen_index]  # modified

                ## concatenate under-sampling
                ## results to modified training set
                data_new = pd.concat([chosen_obs, data_new], ignore_index = True)  # modified

            # added
            ## concatenate 'normal' data
            ## to modified training set
            ## without undersamping
            else:
                original_obs = data.iloc[list(b_index[i].index)]
                data_new = pd.concat([original_obs, data_new], ignore_index = True)


    ## rename feature headers to originals
    data_new.columns = feat_names

    ## restore response variable y to original position
    if y_col < d - 1:
        cols = list(range(d))
        cols[y_col], cols[d - 1] = cols[d - 1], cols[y_col]
        data_new = data_new[data_new.columns[cols]]

    ## restore original data types
    for j in range(d):
        data_new.iloc[:, j] = data_new.iloc[:, j].astype(feat_dtypes_orig[j])

    ## return modified training set
    return data_new

In [ ]:
# def hardness_bin_oversampling(df, target_col, threshold=0.8, bins=60):
#     """
#     Aplica RO apenas nos bins onde a média do IH está entre os top 20%.
#     """
#     y = df[target_col].values
#     X = df.drop(columns=[target_col]).values

#     # Instance Hardness
#     ih_scores = instance_hardness_regression(X, y)

#     # Threshold global (top 20% mais difíceis)
#     global_threshold = np.quantile(ih_scores, threshold)

#     # Gerar bins
#     counts, bin_edges = np.histogram(y, bins=bins)

#     new_df = df.copy()

#     for i in range(len(bin_edges)-1):
#         left = bin_edges[i]
#         right = bin_edges[i+1]

#         mask = (y >= left) & (y < right)
#         bin_data = df[mask]

#         if len(bin_data) == 0:
#             continue

#         ih_bin = ih_scores[mask]
#         mean_ih = np.mean(ih_bin)

#         # Se bin estiver entre os mais difíceis → aplica oversampling
#         if mean_ih >= global_threshold and len(bin_data) > 5:
#             sampled = bin_data.sample(
#                 n=len(bin_data),
#                 replace=True,
#                 random_state=42
#             )
#             new_df = pd.concat([new_df, sampled], ignore_index=True)

#     return new_df

In [25]:

def evaluate_model(X_train, X_test, y_train, y_test, model):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # MSE
    mse = mean_squared_error(y_test, y_pred)

    # mae
    mae = mean_absolute_error(y_test, y_pred)

    return mse, mae


In [26]:
models = {
    "SVR": SVR(),  # não possui random_state
    "NNET":MLPRegressor(hidden_layer_sizes=(50,50), max_iter=1000, random_state=42),
    "XGB":XGBRegressor(n_estimators=100, random_state=42),
    "RF":  RandomForestRegressor(n_estimators=100, random_state=42),
    "BAGGING": BaggingRegressor(random_state=42)
}

In [27]:
wine = pd.read_csv('/content/drive/MyDrive/TCC/dados/wine_quality.csv')
a1 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a1.csv')
a2 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a2.csv')
a3 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a3.csv')
a7 = pd.read_csv('/content/drive/MyDrive/TCC/dados/a7.csv')
acceleration = pd.read_csv('/content/drive/MyDrive/TCC/dados/acceleration.csv')
airfoild = pd.read_csv('/content/drive/MyDrive/TCC/dados/airfoild.csv')
boston = pd.read_csv('/content/drive/MyDrive/TCC/dados/boston.csv')
concreteStrength = pd.read_csv('/content/drive/MyDrive/TCC/dados/concreteStrength.csv')
heat = pd.read_csv('/content/drive/MyDrive/TCC/dados/heat.csv')
sensory = pd.read_csv('/content/drive/MyDrive/TCC/dados/sensory.csv')
analcatdata_apnea3 = pd.read_csv('/content/drive/MyDrive/TCC/dados/analcatdata_apnea3.csv')
available_power = pd.read_csv('/content/drive/MyDrive/TCC/dados/available_power.csv')
cocomo_numeric = pd.read_csv('/content/drive/MyDrive/TCC/dados/cocomo_numeric.csv')
debutanizer = pd.read_csv('/content/drive/MyDrive/TCC/dados/debutanizer.csv')
fuel_consumption_country = pd.read_csv('/content/drive/MyDrive/TCC/dados/fuel_consumption_country.csv')

abalone = pd.read_csv('/content/drive/MyDrive/TCC/dados/abalone.csv')
california = pd.read_csv('/content/drive/MyDrive/TCC/dados/california.csv')
compactiv = pd.read_csv('/content/drive/MyDrive/TCC/dados/compactiv.csv')
cpu_small = pd.read_csv('/content/drive/MyDrive/TCC/dados/cpu_small.csv')
forestFires = pd.read_csv('/content/drive/MyDrive/TCC/dados/forestFires.csv')
kdd_coil_1 = pd.read_csv('/content/drive/MyDrive/TCC/dados/kdd_coil_1.csv')
lungcancer_shedden = pd.read_csv('/content/drive/MyDrive/TCC/dados/lungcancer_shedden.csv')
maximal_torque = pd.read_csv('/content/drive/MyDrive/TCC/dados/maximal_torque.csv')
meta = pd.read_csv('/content/drive/MyDrive/TCC/dados/meta.csv')
mortgage = pd.read_csv('/content/drive/MyDrive/TCC/dados/mortgage.csv')
pdgfr = pd.read_csv('/content/drive/MyDrive/TCC/dados/pdgfr.csv')
space_ga = pd.read_csv('/content/drive/MyDrive/TCC/dados/space_ga.csv')
treasury = pd.read_csv('/content/drive/MyDrive/TCC/dados/treasury.csv')
triazines = pd.read_csv('/content/drive/MyDrive/TCC/dados/triazines.csv')



datasets = {
    "a1": {"data": a1, "target": "a1"},
     "a2": {"data": a2, "target": "a2"},
     "a3": {"data": a3, "target": "a3"},
     "a7": {"data": a7, "target": "a7"},
      "abalone": {"data": abalone, "target": "Rings"},
      "acceleration": {"data": acceleration, "target": "target"},
     "airfoild": {"data": airfoild, "target": "target"},
     "analcatdata_apnea3": {"data": analcatdata_apnea3, "target": "target"},
      "available_power": {"data": available_power, "target": "target"},
   "boston": {"data": boston, "target": "target"},
    #"california": {"data": california, "target": "target"},
    "cocomo_numeric": {"data": cocomo_numeric, "target": "target"},
    "compactiv": {"data": compactiv, "target": "target"},
     "concreteStrength": {"data": concreteStrength, "target": "target"},
     "cpu_small": {"data": cpu_small, "target": "usr"},
     "debutanizer": {"data": debutanizer, "target": "target"},
    "fuel_consumption_country": {"data": fuel_consumption_country, "target": "fuel.consumption.country"},
    "forestFires": {"data": forestFires, "target": "Area"},
     "heat": {"data": heat, "target": "heat"},
     "kdd_coil_1": {"data": kdd_coil_1, "target": "target"},
     "lungcancer_shedden": {"data": lungcancer_shedden, "target": "target"},
      "maximal_torque": {"data": maximal_torque, "target": "maximal.torque"},
    "meta": {"data": meta, "target": "target"},
    "mortgage": {"data": mortgage, "target": "target"},
    "pdgfr": {"data": pdgfr, "target": "target"},
    "sensory": {"data": sensory, "target": "target"},
     "space_ga": {"data": space_ga, "target": "target"},
    "treasury": {"data": treasury, "target": "target"},
    "triazines": {"data": triazines, "target": "target"},
    "wine": {"data": wine, "target": "target"},
}


Teste para GN

In [29]:
from sklearn.model_selection import RepeatedKFold

rkf = RepeatedKFold(n_splits=5, n_repeats=2, random_state=42)

results = []

for name, info in datasets.items():
    df = info["data"].copy()
    target = info["target"]

    print(f"\nProcessando: {name}")

    X = df.drop(columns=[target])
    y = df[target]

    # Loop RepeatedKFold
    for fold_id, (train_idx, test_idx) in enumerate(rkf.split(X)):
        repeat = fold_id // 5 + 1
        fold = fold_id % 5 + 1

        print(f"  Repetição {repeat}/2 - Fold {fold}/5")

        X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
        y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

        base_train = pd.concat([X_train, y_train], axis=1).reset_index(drop=True)

        # Oversampling dentro do fold
        try:
            train_hard = apply_gn_dist(base_train, target)
        except Exception as e:
            print(f"⚠️ GN falhou no fold {fold} da repetição {repeat}: {e}")
            train_hard = base_train.copy()

        versions = {
            "Distance-GN": train_hard
        }

        for model_name, model in models.items():
            for version_name, train_data in versions.items():

                Xtr = train_data.drop(columns=[target])
                ytr = train_data[target]

                Xts = X_test
                yts = y_test

                # Ajuste e predição
                model.fit(Xtr, ytr)
                y_pred = model.predict(Xts)

                # Cálculo das métricas
                mse = mean_squared_error(yts, y_pred)
                mae = mean_absolute_error(yts, y_pred)

                results.append({
                    "Dataset": name,
                    "Repeticao": repeat,
                    "Fold": fold,
                    "Model": model_name,
                    "Versao": version_name,
                    "MSE": mse,
                    "MAE": mae,
                    "y_pred": y_pred.tolist(),   # <<<<<<<< AQUI EXPORTA O Y PREDITO
                    "y_true": yts.tolist()        # (opcional mas recomendado)
                })



Processando: a1
  Repetição 1/2 - Fold 1/5
  Repetição 1/2 - Fold 2/5


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


  Repetição 1/2 - Fold 3/5
  Repetição 1/2 - Fold 4/5


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


  Repetição 1/2 - Fold 5/5


...

  Repetição 2/2 - Fold 4/5


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(


  Repetição 2/2 - Fold 5/5


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (1000) reached and the optimization hasn't converged yet.
  warnings.warn(



Processando: compactiv
  Repetição 1/2 - Fold 1/5
  Repetição 1/2 - Fold 2/5
  Repetição 1/2 - Fold 3/5
  Repetição 1/2 - Fold 4/5
  Repetição 1/2 - Fold 5/5
  Repetição 2/2 - Fold 1/5
  Repetição 2/2 - Fold 2/5
  Repetição 2/2 - Fold 3/5
  Repetição 2/2 - Fold 4/5
  Repetição 2/2 - Fold 5/5

Processando: concreteStrength
  Repetição 1/2 - Fold 1/5
  Repetição 1/2 - Fold 2/5
  Repetição 1/2 - Fold 3/5
  Repetição 1/2 - Fold 4/5
  Repetição 1/2 - Fold 5/5
  Repetição 2/2 - Fold 1/5
  Repetição 2/2 - Fold 2/5
  Repetição 2/2 - Fold 3/5
  Repetição 2/2 - Fold 4/5
  Repetição 2/2 - Fold 5/5

Processando: cpu_small
  Repetição 1/2 - Fold 1/5
  Repetição 1/2 - Fold 2/5
  Repetição 1/2 - Fold 3/5
  Repetição 1/2 - Fold 4/5
  Repetição 1/2 - Fold 5/5
  Repetição 2/2 - Fold 1/5
  Repetição 2/2 - Fold 2/5
  Repetição 2/2 - Fold 3/5
  Repetição 2/2 - Fold 4/5
  Repetição 2/2 - Fold 5/5

Processando: debutanizer
  Repetição 1/2 - Fold 1/5
  Repetição 1/2 - Fold 2/5
  Repetição 1/2 - Fold 3/5
  Re

In [30]:
results_df = pd.DataFrame(results)
results_df.head(50)


,Dataset,Repeticao,Fold,Model,Versao,MSE,MAE,y_pred,y_true
0,a1,1,1,SVR,Distance-GN,3.883463e+02,15.976073,"[24.939856038518087, 24.941017485978872, 19.92...","[17.0, 69.9, 46.2, 50.6, 0.0, 13.0, 3.4, 29.5,..."
1,a1,1,1,NNET,Distance-GN,3.019194e+06,644.671809,"[15.64466761588913, 16.726298729730118, 291.27...","[17.0, 69.9, 46.2, 50.6, 0.0, 13.0, 3.4, 29.5,..."
2,a1,1,1,XGB,Distance-GN,5.047230e+02,18.141357,"[33.67140197753906, 42.70377731323242, 16.6483...","[17.0, 69.9, 46.2, 50.6, 0.0, 13.0, 3.4, 29.5,..."
3,a1,1,1,RF,Distance-GN,3.891469e+02,15.947237,"[37.84553988806531, 38.192917763734876, 46.691...","[17.0, 69.9, 46.2, 50.6, 0.0, 13.0, 3.4, 29.5,..."
4,a1,1,1,BAGGING,Distance-GN,3.872044e+02,15.950594,"[51.20084295926129, 33.94451109385454, 56.9501...","[17.0, 69.9, 46.2, 50.6, 0.0, 13.0, 3.4, 29.5,..."
5,a1,1,2,SVR,Distance-GN,3.928052e+02,15.814758,"[10.543530082873984, 19.10803001022137, 25.692...","[3.3, 15.1, 43.5, 29.7, 33.9, 6.9, 18.3, 66.0,..."
6,a1,1,2,NNET,Distance-GN,1.586749e+05,194.667647,"[195.72460648221914, 240.07003909046017, 54.38...","[3.3, 15.1, 43.5, 29.7, 33.9, 6.9, 18.3, 66.0,..."
7,a1,1,2,XGB,Distance-GN,3.136644e+02,12.282455,"[8.947493553161621, 12.570161819458008, 53.556...","[3.3, 15.1, 43.5, 29.7, 33.9, 6.9, 18.3, 66.0,..."
8,a1,1,2,RF,Distance-GN,2.312140e+02,11.139970,"[10.772283352899542, 24.823653899768537, 48.67...","[3.3, 15.1, 43.5, 29.7, 33.9, 6.9, 18.3, 66.0,..."
9,a1,1,2,BAGGING,Distance-GN,2.962711e+02,11.939518,"[2.7699999999999996, 17.26975009651765, 37.435...","[3.3, 15.1, 43.5, 29.7, 33.9, 6.9, 18.3, 66.0,..."


In [31]:
results_df.to_csv("Distance_GN.csv", index=False)